# Baby Cry Detection with MobileNetV2

This notebook implements a baby cry detection model using transfer learning with MobileNetV2 and MFCC features. The model is trained to classify audio segments as either 'cry' or 'not_cry'.

## 1. Setup and Imports

First, let's import all required libraries and set up our environment.

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import ReduceLROnPlateau
from datetime import datetime
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Model Definition

Define the MobileNetV2-based model architecture for cry detection.

In [ ]:
class MobileNetV2_Crying(nn.Module):
    def __init__(self):
        super(MobileNetV2_Crying, self).__init__()

        # Load pretrained MobileNetV2
        self.model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

        # Modify input layer for 1 channel (MFCC)
        self.model.features[0][0] = nn.Conv2d(
            in_channels=1,      # MFCC has 1 channel
            out_channels=32,
            kernel_size=3,
            stride=2,
            padding=1,
            bias=False
        )

        # Modify output layer for binary classification
        self.model.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(self.model.last_channel, 1)  # Binary classification
        )

    def forward(self, x):
        return self.model(x).squeeze()

## 3. Data Loading

Load and prepare the dataset using the DatasetLoader class.

In [ ]:
from src.utils import DatasetLoader

# Initialize DatasetLoader
data_dir = project_root / 'data/dataset'
processed_dir = project_root / 'data/processed'

dataset_loader = DatasetLoader(data_dir=data_dir, processed_dir=processed_dir)

# Load metadata and display dataset statistics
metadata = dataset_loader.load_metadata()

# Convert audio to MFCC features if not already done
dataset_loader.convert_audio_to_mfcc()

# Prepare dataloaders
batch_size = 32
train_loader, val_loader, test_loader = dataset_loader.prepare_dataset(batch_size=batch_size)

## 4. Training Configuration

In [ ]:
# Training parameters
num_epochs = 50
learning_rate = 0.001

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create directories for logs and checkpoints
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_dir = project_root / 'runs' / timestamp
checkpoint_dir = run_dir / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Initialize tensorboard writer
writer = SummaryWriter(run_dir)

# Get class weights for imbalanced dataset
class_weights = dataset_loader.get_class_weights()
pos_weight = class_weights.to(device) if torch.is_tensor(class_weights) else class_weights

# Initialize model
model = MobileNetV2_Crying()

# Use DataParallel if multiple GPUs available
if device.type == 'cuda' and torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

model = model.to(device)

# Loss function and optimizer
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, 
                             verbose=True, threshold=0.0001, min_lr=1e-6)

## 5. Training Function

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                num_epochs, device, writer, checkpoint_dir):
    best_val_loss = float('inf')
    epochs_no_improve = 0
    early_stop_patience = 10
    
    train_losses = []
    val_losses = []
    learning_rates = []
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        train_preds = []
        train_targets = []
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for batch_idx, (inputs, targets) in enumerate(pbar):
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            predicted = (outputs > 0.5).float()
            train_total += targets.size(0)
            train_correct += (predicted == targets).sum().item()
            
            train_preds.extend(predicted.cpu().detach().numpy())
            train_targets.extend(targets.cpu().detach().numpy())
            
            pbar.set_postfix({
                'loss': train_loss/(batch_idx+1),
                'acc': 100.*train_correct/train_total
            })
        
        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        val_preds = []
        val_targets = []
        
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                
                val_loss += loss.item()
                predicted = (outputs > 0.5).float()
                val_total += targets.size(0)
                val_correct += (predicted == targets).sum().item()
                
                val_preds.extend(predicted.cpu().numpy())
                val_targets.extend(targets.cpu().numpy())
        
        # Calculate metrics
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        train_acc = 100. * train_correct / train_total
        val_acc = 100. * val_correct / val_total
        
        train_precision = precision_score(train_targets, train_preds, zero_division=0)
        train_recall = recall_score(train_targets, train_preds, zero_division=0)
        train_f1 = f1_score(train_targets, train_preds, zero_division=0)
        
        val_precision = precision_score(val_targets, val_preds, zero_division=0)
        val_recall = recall_score(val_targets, val_preds, zero_division=0)
        val_f1 = f1_score(val_targets, val_preds, zero_division=0)
        
        # Store losses and learning rate
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        learning_rates.append(current_lr)
        
        # Update learning rate scheduler
        scheduler.step(avg_val_loss)
        
        # Log metrics
        writer.add_scalar('Loss/train', avg_train_loss, epoch)
        writer.add_scalar('Loss/val', avg_val_loss, epoch)
        writer.add_scalar('Accuracy/train', train_acc, epoch)
        writer.add_scalar('Accuracy/val', val_acc, epoch)
        writer.add_scalar('Precision/train', train_precision, epoch)
        writer.add_scalar('Precision/val', val_precision, epoch)
        writer.add_scalar('Recall/train', train_recall, epoch)
        writer.add_scalar('Recall/val', val_recall, epoch)
        writer.add_scalar('F1-Score/train', train_f1, epoch)
        writer.add_scalar('F1-Score/val', val_f1, epoch)
        writer.add_scalar('Learning_Rate', current_lr, epoch)
        
        # Save checkpoint
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_loss': avg_val_loss,
            'val_acc': val_acc,
            'val_precision': val_precision,
            'val_recall': val_recall,
            'val_f1': val_f1,
            'train_loss': avg_train_loss,
            'train_acc': train_acc,
            'train_precision': train_precision,
            'train_recall': train_recall,
            'train_f1': train_f1,
            'best_val_loss': best_val_loss,
            'learning_rate': current_lr
        }
        
        torch.save(checkpoint, checkpoint_dir / 'last_model.pth')
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(checkpoint, checkpoint_dir / 'best_model.pth')
            print(f'\nNew best model saved! (Val Loss: {avg_val_loss:.4f})')
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f'\nValidation loss did not improve for {epochs_no_improve} epochs')
        
        print(f'\nEpoch {epoch+1}/{num_epochs}:')
        print(f'Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%')
        print(f'Train Precision: {train_precision:.4f}, Recall: {train_recall:.4f}, F1: {train_f1:.4f}')
        print(f'Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%')
        print(f'Val Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}')
        
        if epochs_no_improve >= early_stop_patience:
            print(f'\nEarly stopping triggered after {epoch+1} epochs!')
            break
    
    return train_losses, val_losses, learning_rates

## 6. Train the Model

In [ ]:
try:
    train_losses, val_losses, learning_rates = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        num_epochs=num_epochs,
        device=device,
        writer=writer,
        checkpoint_dir=checkpoint_dir
    )
    
    # Save the loss data
    loss_data = {
        'train_loss': train_losses,
        'val_loss': val_losses,
        'learning_rates': learning_rates
    }
    torch.save(loss_data, checkpoint_dir / 'loss_data.pth')
    
except Exception as e:
    print(f"Error during training: {e}")
    if device.type == 'cuda':
        print("Error might be CUDA-related. Try setting CUDA_LAUNCH_BLOCKING=1")
        torch.cuda.empty_cache()
finally:
    writer.close()

## 7. Visualize Training Results

In [ ]:
# Save the model
model_path = os.path.join(ML_NOTEBOOK_DIR, 'baby_cry_model')
model.save(model_path)
print(f"Model saved to: {model_path}")

# Note: The notebook will be saved in the ML_Notebook directory
print(f"Notebook location: {os.path.join(ML_NOTEBOOK_DIR, 'train_model.ipynb')}")